In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import anndata as ad
import squidpy as sq

# ----------------------------
# Config
# ----------------------------
SIM_DIR = "./sim_datasets_sig0"
RES_DIR = "./results/sim_sig0/moran_geary"
FIG_DIR = "./fig/sim_sig0/moran_geary"
GT_DIR  = "./results/sim_sig0/ground_truth"

os.makedirs(RES_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(GT_DIR, exist_ok=True)

sim_paths = {
    "easy": os.path.join(SIM_DIR, "sim_easy.h5ad"),
    "medium": os.path.join(SIM_DIR, "sim_medium.h5ad"),
    "hard": os.path.join(SIM_DIR, "sim_hard.h5ad"),
}

N_PERMS = 200  # keep consistent with your earlier runs


# ----------------------------
# Helpers
# ----------------------------
def ensure_spatial_graph(sim):
    """Make sure spatial neighbor graph exists."""
    if "spatial_connectivities" not in sim.obsp or "spatial_distances" not in sim.obsp:
        sq.gr.spatial_neighbors(sim, coord_type="generic", spatial_key="spatial")

def find_pval_sim_col(df):
    """
    Prefer permutation p-values for QQ/calibration.
    squidpy provides: pval_sim, pval_z_sim, pval_norm, and *_fdr_bh versions.
    """
    if "pval_sim" in df.columns:
        return "pval_sim"
    # fallback (shouldn't happen with squidpy perms, but keep robust)
    for c in df.columns:
        if "pval" in c.lower():
            return c
    return None

def qq_plot(pvals_by_diff, title, outpath):
    plt.figure(figsize=(6, 6))
    for diff, pvals in pvals_by_diff.items():
        p = np.asarray(pvals, dtype=float)
        p = p[np.isfinite(p)]
        if len(p) == 0:
            continue
        p = np.sort(p)
        n = len(p)
        # better QQ x-axis than linspace(0,1,...)
        exp = (np.arange(1, n + 1) - 0.5) / n
        plt.plot(exp, p, label=diff)

    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("Expected Uniform")
    plt.ylabel("Observed p-values")
    plt.title(title)
    plt.legend()
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close()


# ----------------------------
# Main
# ----------------------------
manifest_rows = []
qq_pvals_moran = {}
qq_pvals_geary = {}

for diff, path in sim_paths.items():
    print(f"\n=== Loading {diff}: {path} ===")
    sim = ad.read_h5ad(path)

    print(f"{diff}: {sim.n_obs} cells × {sim.n_vars} genes")
    print("pattern_type counts:", sim.var["pattern_type"].value_counts().to_dict())
    print("theta:", sim.uns.get("sim_theta", None))

    # ---- Save ground truth (for next script) ----
    gt = sim.var[["is_spatial", "pattern_type"]].copy()
    gt.index = gt.index.astype(str)
    gt.index.name = "gene"
    gt_out = os.path.join(GT_DIR, f"truth_{diff}.csv")
    gt.to_csv(gt_out)
    print("Saved ground truth:", gt_out)

    # ---- Ensure spatial graph ----
    ensure_spatial_graph(sim)

    # ---- Moran ----
    print(f"Running Moran's I ({diff})...")
    sq.gr.spatial_autocorr(sim, mode="moran", n_perms=N_PERMS)
    moran = sim.uns["moranI"].copy()

    # add truth columns for convenience (safe: moran doesn't include them by default)
    moran = moran.join(gt, how="left")
    moran["method"] = "moran"

    moran_out = os.path.join(RES_DIR, f"{diff}_moran_per_gene.csv")
    moran.to_csv(moran_out)
    print("Saved:", moran_out)

    pcol_m = find_pval_sim_col(moran)
    if pcol_m is None:
        print("WARNING: No p-value column found for Moran; QQ will be skipped.")
    else:
        qq_pvals_moran[diff] = moran.loc[moran["pattern_type"] == "null", pcol_m].values

    # ---- Geary ----
    print(f"Running Geary's C ({diff})...")
    sq.gr.spatial_autocorr(sim, mode="geary", n_perms=N_PERMS)
    geary = sim.uns["gearyC"].copy()

    geary = geary.join(gt, how="left")
    geary["method"] = "geary"

    geary_out = os.path.join(RES_DIR, f"{diff}_geary_per_gene.csv")
    geary.to_csv(geary_out)
    print("Saved:", geary_out)

    pcol_g = find_pval_sim_col(geary)
    if pcol_g is None:
        print("WARNING: No p-value column found for Geary; QQ will be skipped.")
    else:
        qq_pvals_geary[diff] = geary.loc[geary["pattern_type"] == "null", pcol_g].values

    # ---- Manifest row ----
    manifest_rows.append({
        "difficulty": diff,
        "sim_path": path,
        "truth_csv": gt_out,
        "moran_csv": moran_out,
        "geary_csv": geary_out,
        "theta": sim.uns.get("sim_theta", None),
        "n_cells": sim.n_obs,
        "n_genes": sim.n_vars,
        "n_perms": N_PERMS,
        "moran_pval_col_used": pcol_m,
        "geary_pval_col_used": pcol_g,
    })

# ---- Save QQ plots ----
qq_moran_path = os.path.join(FIG_DIR, "qq_moran_pval_sim.png")
qq_geary_path = os.path.join(FIG_DIR, "qq_geary_pval_sim.png")

if len(qq_pvals_moran) > 0:
    qq_plot(qq_pvals_moran, "QQ plot of null p-values (Moran, pval_sim)", qq_moran_path)
    print("\nSaved:", qq_moran_path)
else:
    print("\nNo Moran QQ plot generated (missing p-values).")

if len(qq_pvals_geary) > 0:
    qq_plot(qq_pvals_geary, "QQ plot of null p-values (Geary, pval_sim)", qq_geary_path)
    print("Saved:", qq_geary_path)
else:
    print("No Geary QQ plot generated (missing p-values).")

# ---- Save manifest (for next script) ----
manifest = pd.DataFrame(manifest_rows)
manifest_out = os.path.join(RES_DIR, "manifest_moran_geary_truth.csv")
manifest.to_csv(manifest_out, index=False)
print("\nSaved manifest:", manifest_out)

print("\nDONE.")


=== Loading easy: ./sim_datasets_sig0/sim_easy.h5ad ===
easy: 9560 cells × 300 genes
pattern_type counts: {'null': 150, 'gp': 60, 'hotspot': 45, 'stripes': 45}
theta: 5.0
Saved ground truth: ./results/sim_sig0/ground_truth/truth_easy.csv
Running Moran's I (easy)...


/opt/anaconda3/envs/LCL/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/anaconda3/envs/LCL/lib/python3.10/threading.py", line 973, in _bootstrap
    self._bootstrap_inner()
  File "/opt/anaconda3/envs/LCL/lib/python3.10/threadi

Saved: ./results/sim_sig0/moran_geary/easy_moran_per_gene.csv
Running Geary's C (easy)...


/opt/anaconda3/envs/LCL/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/anaconda3/envs/LCL/lib/python3.10/threading.py", line 973, in _bootstrap
    self._bootstrap_inner()
  File "/opt/anaconda3/envs/LCL/lib/python3.10/threadi

Saved: ./results/sim_sig0/moran_geary/easy_geary_per_gene.csv

=== Loading medium: ./sim_datasets_sig0/sim_medium.h5ad ===
medium: 9560 cells × 300 genes
pattern_type counts: {'null': 150, 'gp': 60, 'hotspot': 45, 'stripes': 45}
theta: 5.0
Saved ground truth: ./results/sim_sig0/ground_truth/truth_medium.csv
Running Moran's I (medium)...


...

Saved: ./results/sim_sig0/moran_geary/medium_geary_per_gene.csv

=== Loading hard: ./sim_datasets_sig0/sim_hard.h5ad ===
hard: 9560 cells × 300 genes
pattern_type counts: {'null': 150, 'gp': 60, 'hotspot': 45, 'stripes': 45}
theta: 5.0
Saved ground truth: ./results/sim_sig0/ground_truth/truth_hard.csv
Running Moran's I (hard)...


/opt/anaconda3/envs/LCL/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/anaconda3/envs/LCL/lib/python3.10/threading.py", line 973, in _bootstrap
    self._bootstrap_inner()
  File "/opt/anaconda3/envs/LCL/lib/python3.10/threadi

Saved: ./results/sim_sig0/moran_geary/hard_moran_per_gene.csv
Running Geary's C (hard)...


/opt/anaconda3/envs/LCL/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/anaconda3/envs/LCL/lib/python3.10/threading.py", line 973, in _bootstrap
    self._bootstrap_inner()
  File "/opt/anaconda3/envs/LCL/lib/python3.10/threadi

Saved: ./results/sim_sig0/moran_geary/hard_geary_per_gene.csv

Saved: ./fig/sim_sig0/moran_geary/qq_moran_pval_sim.png
Saved: ./fig/sim_sig0/moran_geary/qq_geary_pval_sim.png

Saved manifest: ./results/sim_sig0/moran_geary/manifest_moran_geary_truth.csv

DONE.
